In [26]:
import logging
import warnings
import os

logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['TQDM_DISABLE'] = '1'

In [1]:
from IPython.display import HTML, display

def set_output_wrap():
    display(HTML('''
    <style>
      pre {
          white-space: pre-wrap !important;
          word-break: break-word !important;
      }
    </style>
    '''))

# Automatically apply CSS before running any cell
get_ipython().events.register('pre_run_cell', set_output_wrap)

# Counterspeech Against Hate and Misinformation: An NLP Perspective

### A Hands-on Tutorial

---

**Presenter:** `Daniel Russo, Fondazione Bruno Kessler`

**Contact:** `drusso@fbk.eu`

**Credits:** `Daniel Russo, Helena Bonaldi, Marco Guerini`

---

## Index

*   [0. Setup](#scrollTo=b3a375b2)
*   [1. Naive counterspeech generation](#scrollTo=9f946d30)
    *   [1.1 What goes wrong](#scrollTo=f2855570)
    *   [1.2 Knowledge-driven generation](#scrollTo=3UVZN_3nQ3gm)
    *   [1.3 Limitations: where does the knowledge come from?](#scrollTo=U4FsHUlNzmcf)
*   [2. RAG](#scrollTo=415ffc6a)
    *   [2.1 Retrieving existing counterspeech examples](#scrollTo=trkfaC_sUyBq)
        *   [2.1.1 Building the counterspeech knowledge base](#scrollTo=66c0a9b7)
        *   [2.1.2 Counterspeech retrieval: sparse vs. dense](#scrollTo=m0AGRX__zmcf)
    *   [2.2 Counterspeech Generation with RAG](#scrollTo=3goND91RVTBC)
        *   [2.2.1 Building the Knowledge Base](#scrollTo=8sG5A0SuVlt1)
        *   [2.2.2 Retrieving the knowledge](#scrollTo=fwopNPE1zmcf)
        *   [2.2.3 Generation](#scrollTo=A3a1P37pzmcg)
    *   [2.3 Challenges and limitations](#scrollTo=6dv_weIEzmcg)
*   [3. Evaluation](#scrollTo=0ezazOqiGo1x)
    *   [3.1 Automatic evaluation: ROUGE and BERTScore](#scrollTo=dI90ajHyReK2)
    *   [3.2 LLM-as-a-judge](#scrollTo=ueDWHM7mzmch)
    *   [3.3 Human evaluation](#scrollTo=laqVmCWYzmci)
*   [4. Conclusions](#scrollTo=yahzK11Z6iql)


## ⚠️ Content warning

This tutorial is about countering hate speech and misinformation, which means **some cells below contain examples of hateful and/or false statements**. They are:

- **Not the presenters' views**
- **Clearly marked** wherever they appear, so you can skip a cell if you'd rather not read the example and just look at the output/discussion instead.

---
## 0. Setup

We'll use a small, openly-licensed instruction-tuned model that runs comfortably on a free Colab GPU (T4), so the whole notebook is reproducible without an API key. `Qwen2.5-3B-Instruct` is the default below because it's ungated on the Hugging Face Hub (no "request access" step) and fast enough for a live session.

**Before running:** in Colab, go to `Runtime → Change runtime type` and select a GPU (T4 is enough).

In [2]:
# Install dependencies (takes ~1-2 minutes on first run)
!pip install -q transformers accelerate sentence-transformers rank_bm25 faiss-cpu \
    rouge-score bert-score ragas datasets huggingface_hub


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go to Runtime > Change runtime type and select a GPU,")
    print("or expect the generation cells below to be slow on CPU.")


# Load the model and tokenizer
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # ungated, ~3B params, fine on a free T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")


def generate(prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

---
## 1. Naive counterspeech generation

LLMs are good at nearly everything nowadays, or at least they seem to be. So the obvious first move is: just ask one to write a counter-narrative, and see what happens.

We'll use three running examples throughout this section, of increasing difficulty:

1. **Hate speech only** : a stereotyping, hateful claim about a target group, of the kind used repeatedly in counterspeech research (modelled on examples from the [MT-CONAN benchmark](https://aclanthology.org/2021.acl-long.250/)).
2. **Misinformation only** : a factual-sounding but false claim, with no explicit hateful language.
3. **Hate speech + misinformation, entangled** : closer to what actually circulates online. This example is taken from a dataset of single-turn counterspeech dialogues against messages that combine hate speech and misinformation ([Martone et al., 2026](https://arxiv.org/abs/2605.22435)).

In [3]:
# ⚠️ Example claims below are shown for illustration only — not the presenters' views.

# 1. Hate speech only
HATE_CLAIM = "Gays keep saying they are normal but then want to be treated in a special way."

# 2. Misinformation only
MISINFO_CLAIM = "A proposed bill in Canada would subject anyone who misgenders others or engages in anti-LGBTQ protests to prosecution and a $25,000 fine."

# 3. Hate speech + misinformation, entangled
HS_MISINFO_CLAIM = "Did you hear that Canada is trying to make it illegal to even protest against LGBTQ stuff? What’s next, you can’t even express your opinion?"

CLAIMS = {
    "HS only": HATE_CLAIM,
    "Misinfo only": MISINFO_CLAIM,
    "HS + misinfo": HS_MISINFO_CLAIM,
}

def naive_prompts(claim):
    return {
        "instructed": f'Generate a short counterspeech in response to the following statement: "{claim}"' }

### 1.1 What goes wrong

#### (a): **Repetitive, formulaic patterns**

Run the *instructed* prompt across all three claims and look only at how each response opens — you'll likely spot the **same stock phrases recurring regardless of what's actually being countered**.

All responses start with *"It's important to..."*.  Moreover, the tendency to include *"I understand your perspective/concern"* is more than stylistically dull: it can read as validating the premise rather than rejecting it, which is the opposite of what a counter-narrative should do.

Most instruction-tuned LLMs are aligned to be safe by default, which pushes them toward vague, hedging language — *"let's embrace…", "we should strive to…", “it is crucial/essential/important to”* — to avoid saying anything that could be misread. [Bonaldi et al. (2024)](https://aclanthology.org/2024.emnlp-main.201/) measured this directly: outputs from "guarded" models used this kind of vague periphrasis about *59% more often*, and scored significantly lower on argumentative cogency than outputs from the same models with guardrails relaxed — while scoring *no differently on safety*. In other words, at least some of what guardrails buy here is vagueness, not safety.

In [7]:
for label, claim in CLAIMS.items():
    prompt = naive_prompts(claim)["instructed"]
    cs = generate(prompt)
    print(f"=== {label} ===\n{cs}\n")
    print()

=== HS only ===
It's important to recognize that everyone has unique experiences and identities, including those who identify as LGBTQ+. The idea of being "treated in a special way" is often misunderstood. Many LGBTQ+ individuals do not seek special treatment; rather, they simply want equal rights, respect, and the opportunity to live their lives without facing discrimination or prejudice. This includes things like the right to marry the person they love, access to healthcare, and protection from hate crimes. Being treated equally means having the same opportunities


=== Misinfo only ===
It's important to consider the broader context and intent behind such legislation. While it's crucial to protect individuals from discrimination and hate speech, it's equally vital to ensure that freedom of expression is not overly restricted. A balanced approach could involve robust protections against harassment and discrimination while safeguarding the right to peaceful protest and free speech. The

#### **(b): Hallucination and factuality**

Now the misinformation and HS+misinfo claims. **Does the model's rebuttal cite any actual numbers or sources? If it does, are they real?** Compare against the real fact-check for the Martone et al. (2026) example, from [Associated Press News](https://apnews.com/article/fact-check-canada-bill-LGBTQ-777684926093).

Countering a stereotype or a false claim requires being right — but an LLM's internal knowledge can be stale, biased toward whatever was common in its training data, or simply wrong about a specific community's history or a specific statistic. This bites hardest on misinformation (where getting the facts right is* the task) but affects hate-speech countering too, whenever a good counter-narrative needs a specific fact the model doesn't reliably know. This is the direct motivation for Section 2: rather than trusting the model's parametric memory, ground its output in retrieved, vetted evidence.

In [8]:
for label in ["Misinfo only", "HS + misinfo"]:
    claim = CLAIMS[label]
    prompt = naive_prompts(claim)["instructed"]
    print(f"=== {label} ===")
    print(f"CLAIM: {claim}\n")
    print("GENERATION:")
    print(generate(prompt))
    print()

=== Misinfo only ===
CLAIM: A proposed bill in Canada would subject anyone who misgenders others or engages in anti-LGBTQ protests to prosecution and a $25,000 fine.

GENERATION:
It's important to consider the broader context and intent behind such legislation. While it's crucial to protect individuals from discrimination and hate speech, it's equally vital to ensure that freedom of expression is not overly restricted. A balanced approach could involve robust protections against harassment and discrimination while safeguarding the right to peaceful protest and free speech. The goal should be to foster an inclusive environment without stifling legitimate discourse.

=== HS + misinfo ===
CLAIM: Did you hear that Canada is trying to make it illegal to even protest against LGBTQ stuff? What’s next, you can’t even express your opinion?

GENERATION:
It's important to approach such statements with careful consideration of the context and specific details. While it's true that some countries h

### Why not just fine-tune?

One natural fix for the problems in 1.1 is to **fine-tune** the model on examples of the counterspeech we actually want — expert-written, argumentative, non-hedgy — so its default style shifts toward that instead of the generic "helpful assistant" register. This is a real, widely-used approach in the counterspeech-generation literature, and it can noticeably improve style and tone.

It comes with real costs, though:

- **Data.** It takes a reasonably large set of high-quality `(claim, counter-narrative)` pairs. Expert-written counterspeech is scarce and expensive to produce, and low-quality training data will just teach the model new bad habits instead of good ones.
- **Cost and maintenance.** Fine-tuning takes compute, and the result is a snapshot in time: as soon as a new strain of misinformation appears, or you want to switch to a newer base model, you're paying that cost again. For larger models this adds up fast.
- **It doesn't fix factuality.** Fine-tuning changes the model's *style*, not its *knowledge*. A fine-tuned model can produce a perfectly-toned, argumentative, non-hedgy response that is still confidently wrong — nothing about fine-tuning on style guarantees the facts inside the response are correct.

That last point is what the rest of this tutorial focuses on. Instead of (or in addition to) retraining the model, we can **give it the right knowledge at inference time**. Two ways to do that, in order of increasing sophistication: hand-fed knowledge (next), and then automatic retrieval — Retrieval-Augmented Generation — in Section 2.


### 1.2 Knowledge-driven generation

Instead of retraining the model, let's just **hand it the right piece of knowledge and tell it to use it**. This is the simplest possible form of grounding: for each claim, we manually pick a short, trustworthy text that addresses it — a stereotype-debunking passage for hate speech, a fact-checking excerpt for misinformation — and put it directly in the prompt.

We reuse the same three claims from Section 1, this time paired with hand-picked knowledge:
- for `HATE_CLAIM`, a short passage from [PG Action](https://www.pgaction.org/inclusion/pdf/myths-v-realities.pdf) addressing the underlying stereotype (the LGBTQIA+ people are requesting a special treatment);
- for `MISINFO_CLAIM`, a fact-checking passage from [Associated Press News](https://apnews.com/article/fact-check-canada-bill-LGBTQ-777684926093) on the alleged $25,000 Canada fine;
- for `HS_MISINFO_CLAIM`, **both** — the stereotype passage *and* a fact-check

#### **CODE — define the knowledge:**

In [9]:
HS_KNOWLEDGE = (
    """
    This is not true. There are no special rights being claimed by or for LGBTI people.
    They are entitled to enjoy the same human rights and fundamental freedoms to which
    every human being is entitled. Regretfully, these rights and freedoms are denied to
    millions of people around the world just because of their sexual orientation and gender
    identity. This is why there is a need to focus on ending discrimination on the basis
    of sexual orientation and gender identity and ensure the inclusion of all LGBTI people in development.
    """
)

MISINFO_KNOWLEDGE = (
    """
    The bill would not institute a blanket ban on misgendering and anti-LGBTQ protests.
    The legislation, introduced by members of the opposition party in the Legislative Assembly of Ontario, would
    allow the province’s attorney general to temporarily prohibit people from engaging in acts of intimidation,
    such as threats or homophobic protests, within 100 meters (328 feet) of certain properties. Proponents say
    that the bill is intended to protect drag performers and LGBTQ communities from intimidation.
    The legislation, which was introduced in the Legislative Assembly of Ontario on April 4, would permit
    the Ontario attorney general to temporarily designate a property as a “2SLGBTQI+ community safety zone”
    and prohibit an “act of intimidation” — such as threats, transphobic or homophobic demonstrations, and distributing
    “hate propaganda” — within 100 meters of that property, according to the bill text.'
    """
)


# For the combined claim, give the model both pieces of knowledge: the fact-check of the specific
# statistic, and the broader stereotype rebuttal — since the claim leans on both.
HS_MISINFO_KNOWLEDGE = HS_KNOWLEDGE + "\n\n" + MISINFO_KNOWLEDGE

KNOWLEDGE = {
    "HS only": HS_KNOWLEDGE,
    "Misinfo only": MISINFO_KNOWLEDGE,
    "HS + misinfo": HS_MISINFO_KNOWLEDGE,
}

#### **CODE — generate with knowledge:**

Now prompt the model with each claim *and* its paired knowledge, and see how the output changes compared to Section 1. You should see the knowledge-grounded versions:

- cite something **specific and checkable**, rather than vague reassurance;
- for the misinformation and HS+misinfo claims in particular, correctly identify with factual information *why* the claim is wrong.

In [11]:
def knowledge_prompt(claim, knowledge):
    return (
        f"Knowledge: {knowledge}\n\n"
        f'Based on the provided knowledge, write a two-sentence response countering the following statement: "{claim}"'
    )

for label, claim in CLAIMS.items():
    prompt = knowledge_prompt(claim, KNOWLEDGE[label])
    print(f"=== {label} ===")
    print(f"CLAIM: {claim}\n")
    print(generate(prompt))
    print()

=== HS only ===
CLAIM: Gays keep saying they are normal but then want to be treated in a special way.

The statement that "gays keep saying they are normal but then want to be treated in a special way" is inaccurate. LGBTI individuals do not claim special rights; rather, they seek equal treatment and protection under the law, ensuring they can enjoy the same human rights and fundamental freedoms as everyone else, regardless of their sexual orientation or gender identity. This includes basic rights such as freedom from discrimination, the right to marry, and access to healthcare, which are essential for full societal inclusion and equality.

=== Misinfo only ===
CLAIM: A proposed bill in Canada would subject anyone who misgenders others or engages in anti-LGBTQ protests to prosecution and a $25,000 fine.

The proposed bill in Ontario does not impose a blanket ban on misgendering or anti-LGBTQ protests. Instead, it allows the province's attorney general to temporarily prohibit acts of in

### 1.3 Limitations: where does the knowledge come from?

This "fix" only worked because *we*, the presenters, already knew which claim was coming and hand-picked the right passage to counter it, offline, before the session. In a real deployment:

- claims arrive as free text, one at a time, and nobody is sitting there matching each one to the right fact-check by hand;
- the knowledge a real system would draw on — fact-checking articles, NGO reports, expert-written counter-narratives — is far too large to paste into every prompt;
- and a claim rarely matches a knowledge source word-for-word, so "just find the matching text" is itself a non-trivial search problem.

What we actually need is a way to **automatically retrieve** the right knowledge for an arbitrary incoming claim out of a (potentially large) knowledge base — which is exactly what Section 2 does.

---
## 2. RAG

Section 1.4 left us with a concrete problem: hand-picking the right knowledge for every incoming claim doesn't scale. **Retrieval-Augmented Generation (RAG)** automates that step — instead of a human matching claims to evidence, a retriever does it, over a knowledge base built once, up front.

Two quick definitions, for anyone new to the term: a **knowledge base (KB)** here is a collection of short texts we trust — expert-written counter-narratives, fact-checking articles, NGO guidance — each turned into a numeric vector (an **embedding**) and stored in a **vector store** for fast similarity search. **Retrieval** is the step of taking a new claim and finding the KB entries most relevant to it.

We'll build two small, parallel knowledge bases and retrieve from each in turn:
- a **counterspeech base** of previously-written `(claim, counter-narrative)` pairs (2.3) — so the system can reuse or adapt an existing response instead of writing one from scratch.
- a **knowledge base** of fact-checking / stereotype-rebuttal passages (2.1, 2.4) — the automatic version of what we hand-picked in Section 1.2;


### 2.1 Retrieving existing counterspeech examples


Instead of retrieving raw evidence, we retrieve an **existing, expert-written counter-narrative** written for a similar past claim, and reuse or adapt it.
To do that, we need to:
1) build the counterspeech knowledge base;
2) retrieve the counterspeech example.

#### 2.1.1 Building the counterspeech knowledge base

In a full pipeline this KB would be built from several different benchmarks (e.g., MT-CONAN / [DIALOCONAN](https://aclanthology.org/2022.emnlp-main.549/), etc.); here's a tiny illustrative version built using different counterspeech examples related to the same target from the dataset by Martone et al. (2026). Each entry gets embedded with a sentence-embedding model and indexed in **FAISS** for fast similarity search.

**CODE:**
1. Collect the claims and corresponding counterspeech examples into a small KB.
2. Embed each claim with `sentence-transformers`.
3. Build a FAISS index (`IndexFlatIP`, i.e. cosine similarity on normalized vectors) over the embeddings. By indexing the KB by its `claim` field, we will be able to retrieve the closest past claim (sparse and dense) and judge whether its counter-narrative would actually be reusable.



In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_dense_index(texts):
    vectors = embedder.encode(texts, normalize_embeddings=True)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(np.array(vectors))
    return index


# A tiny illustrative counterspeech KB — in a full pipeline this would be built from several different datasets.
CLAIM = "Women love to complain about domestic violence just to get attention."

CN_KB = [
    {"claim": "I can't stand when women claim they've been assaulted. They just do it for attention or revenge!",
     "counter_narrative": """It's important to recognize that the vast majority of women who experience assault do not report it due to a fear of not being believed, rather than seeking attention or revenge. False allegations are extremely rare, and by dismissing these experiences, we risk perpetuating a culture that silences survivors and overlooks the real issue of violence against women."""},

    {"claim": "The pay gap is bullshit! Women just like to complain instead of working as hard as men!",
     "counter_narrative": """The gender pay gap is a complex issue, supported by extensive research, showing that women earn significantly less than men for similar work, with an average gap of 31.4% worldwide. It's important to acknowledge the progress made towards gender equality while also recognising that challenges persist, requiring continued advocacy and support for women's rights and opportunities."""},

    {"claim": "Women are using abortion as a form of birth control. There were over 8 million abortions in Britain just this year. That's disgusting!",
     "counter_narrative": """"Abortion is a complex issue influenced by many factors, and the majority of women who have abortions were using contraception at the time they conceived. The circumstances surrounding each pregnancy are unique, and no woman aspires to undergo an abortion; instead, it serves as a necessary option for those facing unplanned pregnancies or medical complications."""},

    {"claim": "Women are just not fit for leadership roles. Look at the number of female MPs compared to men!",
     "counter_narrative": """While the representation of women in leadership roles may still be growing, women are more ambitious than ever and are actively seeking to take on leadership positions in various sectors. The barriers they face, such as the broken rung in promotions, illustrate that the potential for women to lead effectively is not in question; rather, it is society's responsibility to ensure equal opportunities for all."""},
]


cn_claims = [d["claim"] for d in CN_KB]
cn_dense_index = build_dense_index(cn_claims)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### 2.1.2 Counterspeech retrieval: sparse vs. dense

There are two classic families of retrievers, and it's worth trying both so you can *see* the difference rather than just being told about it:

- **Sparse retrieval (e.g. BM25)** scores documents by keyword/lexical overlap with the query. It's fast, cheap, and its decisions are easy to inspect — but it struggles when the query paraphrases the KB entry instead of reusing its wording.
- **Dense retrieval** (what we just built with FAISS) scores documents by similarity in a learned semantic embedding space. It handles paraphrase much better, but is a black box, and its quality depends entirely on how well the embedding model fits this domain.

In production RAG systems the two are often combined, with a **reranker** (a small model that re-scores the top candidates from either retriever more precisely) as a final step — worth mentioning even if we don't have time to implement it live.

In [15]:
# Sparse retrieval
from rank_bm25 import BM25Okapi

def build_bm25_index(texts):
    tokenized = [text.lower().split() for text in texts]
    return BM25Okapi(tokenized)

def bm25_retrieve(query, texts, bm25_index, k=1):
    scores = bm25_index.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]
    return [(texts[i], float(scores[i])) for i in top_idx]

cn_bm25_index = build_bm25_index(cn_claims)

# Dense retrieval
def dense_retrieve(query, texts, index, k=1):
    query_vec = embedder.encode([query], normalize_embeddings=True)
    scores, idx = index.search(np.array(query_vec), k)
    return [(texts[i], float(s)) for i, s in zip(idx[0], scores[0])]

#### **CODE:** Retrieve the closest past claim (sparse and dense) and judge whether its counter-narrative would actually be reusable.


In [19]:
cn_claim_to_narrative = {item["claim"]: item["counter_narrative"] for item in CN_KB}

dense_hit, dense_score = dense_retrieve(CLAIM, cn_claims, cn_dense_index)[0]
bm25_hit, bm25_score = bm25_retrieve(CLAIM, cn_claims, cn_bm25_index)[0]
print(f"=== {label} ===")
print(f"Query: {CLAIM}")
print()
print(f">> Dense top match ({dense_score:.3f}): {dense_hit}")
print()
print(f"CS: {cn_claim_to_narrative[dense_hit]}")
print()
print(f">> BM25  top match ({bm25_score:.3f}): {bm25_hit}")
print()
print(f"CS: {cn_claim_to_narrative[bm25_hit]}")
print()

=== HS + misinfo ===
Query: Women love to complain about domestic violence just to get attention.

>> Dense top match (0.652): I can't stand when women claim they've been assaulted. They just do it for attention or revenge!

CS: It's important to recognize that the vast majority of women who experience assault do not report it due to a fear of not being believed, rather than seeking attention or revenge. False allegations are extremely rare, and by dismissing these experiences, we risk perpetuating a culture that silences survivors and overlooks the real issue of violence against women.

>> BM25  top match (1.215): The pay gap is bullshit! Women just like to complain instead of working as hard as men!

CS: The gender pay gap is a complex issue, supported by extensive research, showing that women earn significantly less than men for similar work, with an average gap of 31.4% worldwide. It's important to acknowledge the progress made towards gender equality while also recognising that ch

### 2.2 Counterspeech Generation with RAG

We now move to the full **counterspeech generation with RAG**: instead of retrieving existing counterspeech examples, we will:

1) build a knowledge base containing excerpts from fact-checking and stereotype-debunking articles;
2) retrieve the relevant knowledge using sparse and dense retrieval;
3) use the automatically retrieved knowledge to generate counterspeech.

#### 2.2.1 Building the Knowledge Base


We use excerpts from fact-checking and stereotype-debunking articles extracted from the dataset by Martone et al. (2026) as our knowledge base. Each entry gets embedded with a sentence-embedding model and indexed in **FAISS** for fast similarity search.

**CODE:**
1. Collect the knowledge passages into a small KB.
2. Embed each one with `sentence-transformers`.
3. Build a FAISS index (`IndexFlatIP`, i.e. cosine similarity on normalized vectors) over the embeddings.


In [20]:
KN_KB = [
  {"target": "muslims",
   "topic": "Knife attacks in London",
   "source": "https://fullfact.org/online/London-muslim-knife-attacks/",
   "text": "There is no evidence to back up the claim about the number of attacks committed by Muslim immigrants, and it is completely implausible. The police and criminal justice system do not collect data on the religion or immigration status of knife crime offenders so it’s impossible to say how many were Muslims or migrants. Data from the Mayor of London’s Office for Policing and Crime found that in 2017 half the knife crime offenders in London were black, Asian or minority ethnic"},

  {"target": "muslims",
   "topic": "Trump did not fire Muslim federal judges",
   "source": "https://www.factcheck.org/2017/08/trump-didnt-fire-14-muslim-judges/",
   "text": "President Donald Trump did not fire 14 Muslim federal judges, as a story circulating on the internet claims. Skeptical Facebook users flagged the story posted on several websites as potentially fake news. The original post was published Aug. 16 on asamericanasapplepie.org. On its “about” page, As American as Apple Pie says it is “here to be your beacon of something you can kinda rely on sometimes but not really.” It rates itself as “S” for satire at the bottom of each page. In other words, it’s a satirical website.There aren’t even 14 Muslim federal judges to fire. In September 2016, former President Barack Obama nominated attorney Abid Riaz Qureshi to serve on the U.S. District Court for the District of Columbia. He reportedly would have been the first American Muslim to become a federal judge, but his nomination wasnever confirmed by the Senate."},

  {"topic": "women",
  "topic": "Family violence myths",
  "source": "https://safeandequal.org.au/understanding-family-violence/myths/#",
  "text": "No one deserves to be abused. The only person responsible for abuse is the person choosing to use violence and abusive behaviour. Any behaviour by someone towards their family member that is abusive –physical, sexual, emotional or financial – and causes them to fear for their safety is against the law. Family violence is often underreported for fear of not being believed. According to the ABS Personal Safety Survey 2016, less than half of the women who experienced violence from a current partner had sought advice or support about the violence. This type of claim is often used to downplay the issue and shift the blame."},

  {"target": "women",
  "topic": "Myths about abortion",
  "source": "https://www.bpas.org/media/2nelmu3y/10-abortion-myths-booklet.pdf",
  "text": "Our abortion rate is almost exactly the same as rates in comparable developed countries such as France and Denmark, and is lower than Sweden’s. These are all countries which expect women to be able to pursue individual goals and ambitions, and plan the timing and size of their families. There is no ‘right’ number of abortions and a variety of factors can determine whether abortion rates rise or fall. With 15.9 abortions per 1,000 women aged 15-44, the abortion rate for England and Wales is currently at its lowest point since 1998. The majority of women who have abortions were using a method of contraception at the time they conceived. No method of contraception is 100% effective. Sometimes it fails and sometimes women - or their partners - fail to use it properly. ‘Fit and forget’ methods such as the implant and the coil are very effective but for some women the side effects (eg pain and bleeding) are intolerable, while convenient appointments for fittings can also be a struggle to obtain. We can certainly do more to improve women’s access to contraception and in the longer-term develop new and better methods, but abortion will always need to be there as a back-up for women."},

  {"target": "lgbtqia+",
  "topic": "Proposed LGBTQ ‘safety zone’",
  "source": "https://apnews.com/article/fact-check-canada-bill-LGBTQ-777684926093",
  "text": "AP’S ASSESSMENT: False. The bill would not institute a blanket ban on misgendering and anti-LGBTQ protests. The legislation, introduced by members of the opposition party in the Legislative Assembly of Ontario, would allow the province’s attorney general to temporarily prohibit people from engaging in acts of intimidation, such as threats or homophobic protests, within 100 meters (328 feet) of certain properties. Proponents say that the bill is intended to protect drag performers and LGBTQ communities from intimidation. The legislation, which was introduced in the Legislative Assembly of Ontario on April 4, would permit the Ontario attorney general to temporarily designate a property as a “2SLGBTQI+ community safety zone” and prohibit an “act of intimidation” — such as threats, transphobic or homophobic demonstrations, and distributing “hate propaganda” — within 100 meters of that property, according to the bill text.'"},

  {"target": "lgbtqia+",
  "topic": "Pediatric Transgender Clinics",
  "source": "https://leadstories.com/hoax-alert/2023/07/fact-check-disney-is-not-opening-pediatric-transgender-clinics.html",
  "text": """Does this video prove The Walt Disney Company plans to open "Disney-themed pediatric transgender clinics"? No, that's not true: There is no evidence that Disney has announced such a plan. The man who makes the claim admitted to posing as a Disney executive under the guise of "satire" in his YouTube video\\\'s description. There is nothing aside from that satirical video to substantiate the claim. News does not show any credible media reports of Disney announcing a pediatric transgender clinic."""}
]

kn_texts = [d["text"] for d in KN_KB]
kn_dense_index = build_dense_index(kn_texts)

#### 2.2.2 Retrieving the knowledge

Now for each claim, does automatic retrieval find the correct passage?


In [22]:
text_to_topic = {item["text"]: item["topic"] for item in KN_KB}
kn_bm25_index = build_bm25_index(kn_texts)


CLAIMS = {
  "muslims": "Did you know that 11,000 of the 13,000 knife attacks in London were committed by Muslim immigrants? We let them in and this is how they pay us back!",
  "women": "Women love to complain about domestic violence just to get attention. Sometimes they need to be put in their place: it’s not abuse, it’s discipline.",
  "lgbt+": "Did you hear that Canada is trying to make it illegal to even protest against LGBTQ stuff? What’s next, you can’t even express your opinion?"
}

for label, claim in CLAIMS.items():
    dense_hit, dense_score = dense_retrieve(claim, kn_texts, kn_dense_index)[0]
    bm25_hit, bm25_score = bm25_retrieve(claim, kn_texts, kn_bm25_index)[0]
    print(f"=== {label} ===")
    print(f"Query: {claim}")
    print(f"Dense top match ({dense_score:.3f}): {text_to_topic[dense_hit]}")
    # print(f"\n{dense_hit}\n")
    print(f"BM25  top match ({bm25_score:.3f}): {text_to_topic[bm25_hit]}")
    # print(f"\n{bm25_hit}\n")
    print()

=== muslims ===
Query: Did you know that 11,000 of the 13,000 knife attacks in London were committed by Muslim immigrants? We let them in and this is how they pay us back!
Dense top match (0.729): Knife attacks in London
BM25  top match (13.563): Knife attacks in London

=== women ===
Query: Women love to complain about domestic violence just to get attention. Sometimes they need to be put in their place: it’s not abuse, it’s discipline.
Dense top match (0.530): Family violence myths
BM25  top match (6.390): Myths about abortion

=== lgbt+ ===
Query: Did you hear that Canada is trying to make it illegal to even protest against LGBTQ stuff? What’s next, you can’t even express your opinion?
Dense top match (0.554): Proposed LGBTQ ‘safety zone’
BM25  top match (8.609): Trump did not fire Muslim federal judges



#### 2.2.3 Counterspeech Generation

Put retrieval and generation together: retrieve the top knowledge passage for a claim, then reuse `knowledge_prompt()` from Section 1.2 — except now the knowledge is found automatically, not hand-picked.


In [23]:
def rag_generate(claim, texts, index, k=1):
    retrieved = dense_retrieve(claim, texts, index, k=k)
    knowledge = " ".join(text for text, _ in retrieved)
    prompt = knowledge_prompt(claim, knowledge)
    return generate(prompt), knowledge

for label, claim in CLAIMS.items():
    generation, knowledge = rag_generate(claim, kn_texts, kn_dense_index)
    print(f"=== {label} ===")
    print(f"Retrieved: {text_to_topic[knowledge]}\n{knowledge[:90]}...\n")
    print("GENERATION:")
    print(generation)
    print()

=== muslims ===
Retrieved: Knife attacks in London
There is no evidence to back up the claim about the number of attacks committed by Muslim ...

GENERATION:
The claim that 11,000 out of 13,000 knife attacks in London were committed by Muslim immigrants has no evidence to support it. In fact, data from the Mayor of London's Office for Policing and Crime shows that in 2017, half of the knife crime offenders in London were from Black, Asian, or Minority Ethnic backgrounds, which includes various religious groups. Therefore, this statistic does not accurately represent the demographics of those involved in knife

=== women ===
Retrieved: Family violence myths
No one deserves to be abused. The only person responsible for abuse is the person choosing...

GENERATION:
Domestic violence is never acceptable and no one deserves to be abused. It is crucial to recognize that any behavior causing fear for safety, regardless of whether it involves physical, sexual, emotional, or financial abuse, is 

### 2.3 Challenges and limitations

Retrieval quality depends entirely on the query looking enough like the KB entry, in whatever space the retriever uses (keyword overlap for BM25, semantic embedding for dense). Two things break that assumption constantly in the wild:

- **Escalation.** The same underlying claim, made more aggressively, is still the same claim to a fact-checker — but it can shift far enough in embedding/lexical space that retrieval picks a worse match, or none.
- **Register/style shift.** Claims on social media rarely read like a fact-checker's summary of them: shorter, more casual, emoji, hashtags, non-standard spelling. **VerMouth** (Russo et al., EMNLP 2023, ["Countering Misinformation via Emotional Response Generation"](https://aclanthology.org/2023.emnlp-main.703/)) was built precisely because this gap matters: it systematically rewrites claim–counter-narrative pairs into social-media-platform style (and adds an emotional dimension), so generation models are trained on text that actually resembles what they'll see in production, rather than on clean fact-checker prose.

**CODE:** rewrite one of the claims in two ways — more hateful, and in social-media style — and rerun dense retrieval against the knowledge base to see the effect.


In [24]:
# ⚠️ Illustrative paraphrases of a claim, for demonstration only — not the presenters' views.
CLAIM = CLAIMS['muslims']

CLAIM_MORE_HATEFUL = ("Here we go again... fucking muslims are just coming here to slaughter our nationals, when will we send them back in their shithole countries?")

# Same claim, rewritten in social-media-platform (SMP) style — the kind of transformation
CLAIM_SMP_STYLE = (
    "stg like 8 out of 10 stabbings in LDN are done by foreign arrivals 💀 invited em over just to deal with this daily")

for label, claim in [
    ("original", CLAIM),
    ("more hateful", CLAIM_MORE_HATEFUL),
    ("SMP style", CLAIM_SMP_STYLE),
]:
    hit, score = dense_retrieve(claim, kn_texts, kn_dense_index)[0]
    print(f"=== {label} ===")
    print(f"Query: {claim}")
    print(f"Dense top match ({score:.3f}): {text_to_topic[hit]}")
    bm25_hit, bm25_score = bm25_retrieve(claim, kn_texts, kn_bm25_index)[0]
    print(f"BM25  top match ({bm25_score:.3f}): {text_to_topic[bm25_hit]}")
    print()

=== original ===
Query: Did you know that 11,000 of the 13,000 knife attacks in London were committed by Muslim immigrants? We let them in and this is how they pay us back!
Dense top match (0.729): Knife attacks in London
BM25  top match (13.563): Knife attacks in London

=== more hateful ===
Query: Here we go again... fucking muslims are just coming here to slaughter our nationals, when will we send them back in their shithole countries?
Dense top match (0.446): Knife attacks in London
BM25  top match (7.491): Myths about abortion

=== SMP style ===
Query: stg like 8 out of 10 stabbings in LDN are done by foreign arrivals 💀 invited em over just to deal with this daily
Dense top match (0.290): Knife attacks in London
BM25  top match (4.275): Myths about abortion



---
## 3.Evaluation

Ok, we've got several ways to generate counterspeech now — naive, knowledge-driven, RAG. But how do we actually tell whether the output is any good? We'll look at three levels, roughly in order of how much they tell you versus how much they cost to run:

- **Automatic metrics** (ROUGE, BERTScore) — cheap, fast, fully reproducible. We'll see shortly that they're also fairly blunt instruments for this task.
- **LLM-as-a-judge** — more expensive, less reproducible, but can actually reason about *why* a response is good or bad rather than just measuring overlap with one reference.
- **Human evaluation** — the closest thing to a gold standard for properties like cogency and appropriateness, and still not a solved problem.


### 3.1 Automatic evaluation: ROUGE and BERTScore

Both metrics compare a generated response against a **gold reference** — an expert-written counter-narrative for the same claim:

- **ROUGE** counts n-gram overlap (shared words/phrases) between candidate and reference.
- **BERTScore** compares them in embedding space instead of surface words, so it should, in principle, reward paraphrase and penalize it less harshly than ROUGE.

**CODE:**
1. Take one of the claims and its gold reference.
2. Score our RAG generation (Section 2.2.3) against that gold reference.
3. Now repeat the scoring with a **different, hand-written response that is also a good, valid answer to the same claim** — just phrased differently and attacking a different part of it — and see what the metrics say about it.


In [27]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

CLAIM = CLAIMS['women']
# The RAG generation from Section 2.5
CANDIDATE_CS, _ = rag_generate(CLAIM, kn_texts, kn_dense_index)

GOLD_CS = ("""It is crucial to recognize that no one deserves to be subjected to violence, and the blame for abuse lies solely with the abuser. Many survivors of domestic violence face significant barriers in reporting their experiences, as fear of disbelief often silences their voices, and it is essential to support and believe those who come forward with their stories.""")

ALT_VALID_CS = ("""The statement reduces a complicated social issue to an insulting generalization. Personal conflicts should not be justified by claiming that one person has authority over another. Disagreements between people can be handled without humiliation or coercion, and treating harmful behaviour as acceptable simply because of a relationship between the individuals involved is an unreasonable standard. Respect and personal autonomy should apply within households just as they do elsewhere.""")


rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def report(name, candidate, reference):
    r = rouge.score(reference, candidate)
    # Pass verbose=False to keep BERTScore quiet
    _, _, bf = bertscore([candidate], [reference], lang="en", verbose=False)
    print(f"{name}")
    print(f"  ROUGE-1: {r['rouge1'].fmeasure:.3f}   ROUGE-L: {r['rougeL'].fmeasure:.3f}   BERTScore-F1: {bf.item():.3f}")
    print()

# print(f"GOLD reference: {GOLD_CS}\n")
# print(f"Model generation: {CANDIDATE_CS}\n")
# print(f"Alternative valid counter-narrative: {ALT_VALID_CS}\n")

report("Model generation (Section 2.5, RAG)", CANDIDATE_CS, GOLD_CS)
report("Alternative valid counter-narrative (hand-written)", ALT_VALID_CS, GOLD_CS)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Model generation (Section 2.5, RAG)
  ROUGE-1: 0.366   ROUGE-L: 0.239   BERTScore-F1: 0.879



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Alternative valid counter-narrative (hand-written)
  ROUGE-1: 0.173   ROUGE-L: 0.142   BERTScore-F1: 0.841



**What just happened.** `ALT_VALID_CS` is a genuinely good counter-narrative — factually sound, on-topic, well-argued. But because it shares little vocabulary with `GOLD_CS` and has a slightly different argumentation, it will typically score no better — often worse — than the model's own generation, even where the model's generation is weaker in substance.

This is the core limitation: ROUGE and BERTScore measure similarity **to one specific reference**, not "is this a good response to the claim." Counterspeech, like translation or summarization, has many valid answers, and a single gold reference can only reward the one path it happens to represent. This is a large part of why the counterspeech literature increasingly turns to **reference-free methods** — LLM-as-a-judge and human evaluation — for anything beyond a first sanity check.


### 3.2 LLM-as-a-judge

Instead of comparing to a fixed reference, ask a capable LLM to rate the response directly against the *claim*, on whatever criteria we care about. This is **reference-free: it can correctly recognize a valid response even if it looks nothing like our one gold example.**

In production you'd typically use a strong, separate model as the judge (to reduce the risk of a model favouring its own outputs). For this demo we reuse our local model for simplicity — treat the scores as illustrative, not authoritative.

**CODE:** ask the model to rate both `CANDIDATE_CS` and `ALT_VALID_CS` against `HATE_CLAIM` on relevance, groundedness, and tone.


In [28]:
def judge_prompt(claim, response):
    return (
        "You are evaluating a counter-narrative written in response to a hateful or false claim.\n\n"
        f'Claim: "{claim}"\n'
        f'Response: "{response}"\n\n'
        "On a scale of 1 (very poor) to 5 (excellent), rate the response on:\n"
        "1. Relevance: does it directly address the claim?\n"
        "2. Groundedness: does it rely on a specific, checkable fact or argument, rather than vague reassurance?\n"
        "3. Tone: is it firm but non-inflammatory?\n\n"
        "Give a score 1-5 for each criterion, and one sentence of justification."
    )

for name, response in [
    ("Model generation (Section 2.5, RAG)", CANDIDATE_CS),
    ("Alternative valid counter-narrative", ALT_VALID_CS),
]:
    print(f"=== {name} ===\n")
    print(generate(judge_prompt(CLAIM, response), max_new_tokens=200))
    print()

=== Model generation (Section 2.5, RAG) ===

1. **Relevance**: 5  
   Justification: The response directly addresses the claim by refuting the notion that women who report domestic violence are doing so for attention and instead emphasizes the seriousness and illegality of such behavior.

2. **Groundedness**: 4  
   Justification: While the response provides a clear definition of what constitutes domestic violence and its illegality, it could benefit from more specific examples or studies that support the idea that claims of domestic violence are not made for attention. However, the core points are grounded in legal and ethical principles.

3. **Tone**: 5  
   Justification: The tone is firm yet non-inflammatory, maintaining a respectful and assertive stance against the false claim without resorting to anger or personal attacks.

=== Alternative valid counter-narrative ===

1. **Relevance**: 5  
   Justification: The response directly addresses the claim by challenging the notion that 

Compare these judgments with the ROUGE/BERTScore numbers above: a reasonable judge should now rate `ALT_VALID_CS` as good, or better, on relevance and groundedness — unlike the reference-based metrics, which had no way to reward it. That's the whole point of reference-free evaluation.

**It isn't free of problems**, though: LLM judges can be inconsistent across runs, biased toward longer or more confident-sounding answers regardless of actual quality, and — since a judge is itself an LLM — capable of confidently endorsing a factually wrong response as "grounded" if it merely *sounds* well-supported. Treat it as a stronger signal than n-gram overlap, not as ground truth.


### 3.3 Human evaluation

For properties that matter most in this task — is the argument actually *cogent*? does it feel appropriate to the specific community being targeted? would it plausibly change anyone's mind, or just look good on paper? — human judgment is still the closest thing to a gold standard, and most published counterspeech work reports it alongside (or instead of) automatic metrics.

It has real limits of its own, though:

- **Cost and scale.** Human annotation doesn't run in a for-loop; every claim/response pair costs real annotator time, which caps how much you can evaluate.
- **Whose judgment?** A response can read as appropriate to one annotator and tone-deaf to another — ideally, at least some annotators are drawn from the community actually targeted by the claim, which is often skipped for cost or logistical reasons.
- **Annotator fatigue and blind spots.** Read enough LLM-generated counter-narratives in a row and the same repetitive patterns from Section 1.1(a) become easy to stop noticing — annotators habituate just like the model repeats.
- **It still doesn't measure real-world efficacy.** Even a well-annotated "this is a good counter-narrative" score doesn't tell you whether it would actually change a reader's mind, reduce engagement with the hateful content, or hold up in a live back-and-forth — that requires a different kind of study entirely, and is close to an open problem in the field.

**Take-home message for Evaluation.** No single method here is sufficient on its own. Automatic metrics are cheap sanity checks with a real blind spot for valid-but-different answers; LLM-as-a-judge is more flexible but self-referential and inconsistent; human evaluation is the strongest signal but the most expensive and hardest to scale. In practice, robust evaluation of counterspeech generation combines more than one of these.


## 4. Conclusions


This tutorial demonstrated that while large language models (LLMs) are powerful for generating text, **simply asking them to create counterspeech (naive generation) often leads to vague, repetitive, or even factually incorrect responses**.

We explored how grounding LLM generations with external, verified knowledge—first through **hand-picked passages and then via Retrieval-Augmented Generation (RAG) improves the specificity, factuality, and argumentative cogency of the counterspeech**.

Finally, we examined the challenges of evaluating generated counterspeech, highlighting the limitations of automatic metrics like ROUGE and BERTScore, and the strengths and weaknesses of LLM-as-a-judge and human evaluation, emphasizing the **need for a multi-faceted approach to assessment**.